**Objetivo:** Carregar os dados brutos do SIH e CNES, filtrar internações por IAM (CID I21), realizar limpeza e integrar as duas bases.

**Outputs gerados:**
- `data/interim/sih_iam.parquet` — internações IAM limpas
- `data/interim/cnes_hospitais.parquet` — dados hospitalares consolidados
- `data/processed/base_modelagem.parquet` — base final (SIH + CNES)

## Configuração do ambiente

In [1]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path
import sys
import json

In [2]:
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(f"Raiz do projeto: {ROOT}")

Raiz do projeto: /home/carolina/Documents/TCC Documentos/TCC


In [3]:
#pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

#  Caminhos 
RAW_SIH     = Path(ROOT,'data/input/SIH')
RAW_CNES    = Path(ROOT,'data/input/CNES')
INTERIM     = Path(ROOT,'data/interim')
PROCESSED   = Path(ROOT,'data/processed')

INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print('✓ Configurações carregadas')
print(f'  SIH  → {RAW_SIH}')
print(f'  CNES → {RAW_CNES}')

✓ Configurações carregadas
  SIH  → /home/carolina/Documents/TCC Documentos/TCC/data/input/SIH
  CNES → /home/carolina/Documents/TCC Documentos/TCC/data/input/CNES


## Carregando base SIH

In [4]:
arquivos_sih = sorted(glob.glob(str(RAW_SIH / '*.csv')))

Para fazer o primeiro teste iremos apenas utilizar o mes de janheiro de 2025 para continuar a exploração.

### 1. Leitura e Padronização de Colunas
Pegamos o mes de janeiro de 2025 para realizar a leitura e padronizar as colunas com base no dicionário fornecido na pasta `data/external/dicionario_SIH.json`.

In [5]:
path_dicionario =  Path(ROOT,'data/external/dicionario_SIH.json')
path_dicionario
with open(path_dicionario, "r", encoding="utf-8") as f:
    schema = json.load(f)

In [6]:
# criando objeto para renomear colunas
rename_dict = {
    col["old_name"]: col["new_name"]
    for col in schema
}


In [7]:
# carregando um mes de SIH 
df_sih = pd.read_csv(arquivos_sih[0])
df_sih.head(5)

/tmp/ipykernel_331523/1427407375.py:2: DtypeWarning: Columns (0: ETNIA, 1: AUD_JUST, 2: SIS_JUST, 3: DIAGSEC2, 4: DIAGSEC3, 5: DIAGSEC4, 6: DIAGSEC5, 7: DIAGSEC6, 8: DIAGSEC7, 9: DIAGSEC8, 10: DIAGSEC9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sih = pd.read_csv(arquivos_sih[0])


,UF_ZI,ANO_CMPT,MES_CMPT,ESPEC,CGC_HOSP,N_AIH,IDENT,CEP,MUNIC_RES,NASC,...,DIAGSEC9,TPDISEC1,TPDISEC2,TPDISEC3,TPDISEC4,TPDISEC5,TPDISEC6,TPDISEC7,TPDISEC8,TPDISEC9
0,350000,2025,1,1,46374500028366.00,3525100117847,1,11704840,354100,19840716,...,NaN,1,0,0,0,0,0,0,0,0
1,350000,2025,1,2,46374500028366.00,3524130275908,1,11741802,352210,20070606,...,NaN,1,1,1,1,1,0,0,0,0
2,350000,2025,1,2,46374500028366.00,3524130278427,1,11730000,353110,20030120,...,NaN,1,1,1,0,0,0,0,0,0
3,350000,2025,1,2,46374500028366.00,3524130278449,1,11743250,352210,20030405,...,NaN,1,1,1,1,0,0,0,0,0
4,350000,2025,1,2,46374500028366.00,3524130278526,1,11730000,353110,20050224,...,NaN,1,1,1,1,0,0,0,0,0


In [8]:
# renomear colunas de df
df_sih = df_sih.rename(columns=rename_dict)
df_sih.head(5)

,municipio_gestor,ano_competencia,mes_competencia,especialidade_leito,cnpj_hospital,numero_aih,tipo_aih,cep_paciente,municipio_residencia,data_nascimento,...,diagnostico_secundario_9,tipo_diag_sec_1,tipo_diag_sec_2,tipo_diag_sec_3,tipo_diag_sec_4,tipo_diag_sec_5,tipo_diag_sec_6,tipo_diag_sec_7,tipo_diag_sec_8,tipo_diag_sec_9
0,350000,2025,1,1,46374500028366.00,3525100117847,1,11704840,354100,19840716,...,NaN,1,0,0,0,0,0,0,0,0
1,350000,2025,1,2,46374500028366.00,3524130275908,1,11741802,352210,20070606,...,NaN,1,1,1,1,1,0,0,0,0
2,350000,2025,1,2,46374500028366.00,3524130278427,1,11730000,353110,20030120,...,NaN,1,1,1,0,0,0,0,0,0
3,350000,2025,1,2,46374500028366.00,3524130278449,1,11743250,352210,20030405,...,NaN,1,1,1,1,0,0,0,0,0
4,350000,2025,1,2,46374500028366.00,3524130278526,1,11730000,353110,20050224,...,NaN,1,1,1,1,0,0,0,0,0


Agora iremos retirar as colunas não necessarias

In [9]:
cols_keep = [
    "especialidade_leito",
    "numero_aih",
    "data_nascimento",
    "sexo",
    "uti_mes_total",
    "tipo_uti",
    "procedimento_solicitado",
    "procedimento_realizado",
    "data_internacao",
    "data_saida",
    "diagnostico_principal",
    "diagnostico_secundario",
    "motivo_saida",
    "codigo_idade",
    "idade",
    "dias_permanencia",
    "indicador_obito",
    "carater_internacao",
    "cid_notificacao",
    "cnes",
    "cid_associado",
    "cid_morte",
    "complexidade",
    "raca_cor",
    "etnia",
]

# filtrando por colunas
df_sih = df_sih[cols_keep]
df_sih

,especialidade_leito,numero_aih,data_nascimento,sexo,uti_mes_total,tipo_uti,procedimento_solicitado,procedimento_realizado,data_internacao,data_saida,...,dias_permanencia,indicador_obito,carater_internacao,cid_notificacao,cnes,cid_associado,cid_morte,complexidade,raca_cor,etnia
0,1,3525100117847,19840716,3,0,0,407040129,407040129,20250116,20250117,...,1,0,1,NaN,2087804,0,0,2,1,0
1,2,3524130275908,20070606,3,0,0,310010039,310010039,20241205,20241208,...,3,0,2,NaN,2087804,0,0,2,3,0
2,2,3524130278427,20030120,3,0,0,310010039,310010039,20241212,20241216,...,4,0,2,NaN,2087804,0,0,2,1,0
3,2,3524130278449,20030405,3,0,0,310010039,310010039,20241212,20241217,...,5,0,2,NaN,2087804,0,0,2,1,0
4,2,3524130278526,20050224,3,0,0,310010039,310010039,20241212,20241216,...,4,0,2,NaN,2087804,0,0,2,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238139,1,3525109544231,19700728,3,0,0,409070050,409070050,20250121,20250123,...,2,0,1,NaN,2087618,0,0,2,1,0
238140,2,3525109544924,19960528,3,0,0,310010039,310010039,20250113,20250114,...,1,0,2,NaN,2087618,0,0,2,3,0
238141,2,3525109544957,20100415,3,0,0,310010039,310010039,20250116,20250118,...,2,0,2,NaN,2087618,0,0,2,3,0
238142,2,3525109544979,19880921,3,0,0,310010039,310010039,20250111,20250113,...,2,0,2,NaN,2087618,0,0,2,3,0


### 2. Filtro CID (Internações por IAM)
Vamos filtrar as internações primárias e secundárias cujo diagnóstico corresponda a Infarto Agudo do Miocárdio (prefixos `I21` do CID-10). Com isso, focamos apenas na amostra requerida em nosso estudo.

In [10]:
df_sih["diagnostico_principal"] = df_sih["diagnostico_principal"].astype(str)
df_sih["diagnostico_secundario"] = df_sih["diagnostico_secundario"].astype(str)


df_iam = df_sih[
    df_sih["diagnostico_principal"].str.startswith("I21") |
    df_sih["diagnostico_secundario"].str.startswith("I21")
]
df_iam

,especialidade_leito,numero_aih,data_nascimento,sexo,uti_mes_total,tipo_uti,procedimento_solicitado,procedimento_realizado,data_internacao,data_saida,...,dias_permanencia,indicador_obito,carater_internacao,cid_notificacao,cnes,cid_associado,cid_morte,complexidade,raca_cor,etnia
183,1,3524128896860,19740525,1,3,86,406030049,406030049,20241217,20241219,...,2,0,2,NaN,2077396,0,0,3,3,0
184,1,3524128896871,19631015,3,0,0,406030030,406030030,20241211,20241212,...,1,0,2,NaN,2077396,0,0,3,3,0
195,1,3524131709505,19550715,3,2,86,406030022,406030022,20241226,20241228,...,2,1,2,NaN,2077396,0,0,3,1,0
205,1,3524128891019,19510501,3,2,86,406030049,406030049,20241212,20241215,...,3,0,2,NaN,2077396,0,0,3,2,0
207,1,3524128891437,19600521,1,5,86,406030049,406030049,20241213,20241217,...,4,0,2,NaN,2077396,0,0,3,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
237663,3,3524132021510,19360415,1,0,0,303060190,303060190,20250113,20250123,...,10,0,2,NaN,2699915,0,0,2,1,0
237678,3,3524132019287,19560320,1,0,0,301060088,301060088,20241231,20250101,...,1,1,2,NaN,2699915,0,0,2,3,0
237686,3,3524132021982,19731105,3,0,0,303060190,303060190,20250112,20250122,...,10,0,2,NaN,2699915,0,0,2,3,0
237960,3,3525109543461,19590614,1,14,75,303060190,303060190,20241220,20250120,...,31,0,2,NaN,2087618,0,0,2,3,0


### 3. Tratamento de Valores em Branco / Nulos
Em registros administrativos do DataSUS, dados em branco muitas vezes aparecem como strings vazias ou zeros. Vamos padronizá-los como nulos (`pd.NA`) para análise realista da completude do banco.

In [11]:
df_iam = df_iam.replace(["", "0000", "0", "000"], pd.NA)

In [12]:
nulls = pd.DataFrame({
    "qtd_nulos": df_iam.isnull().sum(),
    "perc_nulos": df_iam.isnull().mean() * 100
}).sort_values(by="perc_nulos", ascending=False)

nulls

,qtd_nulos,perc_nulos
diagnostico_secundario,4110,100.00
cid_notificacao,4110,100.00
etnia,65,1.58
numero_aih,0,0.00
uti_mes_total,0,0.00
tipo_uti,0,0.00
data_nascimento,0,0.00
sexo,0,0.00
especialidade_leito,0,0.00
data_internacao,0,0.00


### Salvar a Base Parcial (SIH IAM)
Exportaremos a base limpa unicamente do SIH para possibilitar o recarregamento rápido.

In [13]:
path_sih_interim = INTERIM / "sih_iam.csv"
df_iam.to_csv(path_sih_interim, index=False)
print(f"SIH limpa salva em: {path_sih_interim}")

SIH limpa salva em: /home/carolina/Documents/TCC Documentos/TCC/data/interim/sih_iam.csv


## 4. Integração das Bases do CNES com Dicionários Próprios
Vamos carregar a base de Estabelecimentos (ST) e anexar resumos de Leitos (LT), Equipamentos (EQ), Serviços (SR) e Habilitações (HB).

In [14]:
import glob
import json

def load_cnes_custom(prefix):
    path_dict = Path(ROOT, f'data/external/dicionario_CNES_{prefix.upper()}.json')
    if not path_dict.exists():
        print(f"Dicionário não encontrado para {prefix.upper()}")
        return pd.DataFrame()
        
    with open(path_dict, 'r', encoding='utf-8') as f:
        schema = json.load(f)
        
    rename_dict = {col["old_name"]: col["new_name"] for col in schema}
    cols_keep = [col["new_name"] for col in schema]
    
    arquivos = sorted(glob.glob(str(RAW_CNES / f'{prefix}sp*.csv')))
    if not arquivos: return pd.DataFrame()
    
    # Lê o mais recente
    df = pd.read_csv(arquivos[-1], sep=',', encoding='latin-1', dtype=str, on_bad_lines='skip')
    
    # Renomeia
    df = df.rename(columns=rename_dict)
    
    # Filtra pelas necessarias presentes
    cols_presentes = [c for c in cols_keep if c in df.columns]
    return df[cols_presentes]

df_st = load_cnes_custom('st')
df_lt = load_cnes_custom('lt')
df_eq = load_cnes_custom('eq')
df_sr = load_cnes_custom('sr')
df_hb = load_cnes_custom('hb')

print(f"ST: {df_st.shape[0]} | LT: {df_lt.shape[0]} | EQ: {df_eq.shape[0]} | SR: {df_sr.shape[0]} | HB: {df_hb.shape[0]}")


ST: 109849 | LT: 8350 | EQ: 242363 | SR: 177050 | HB: 6755


### Consolidação na base ST
Agrupando as extensões por `codigo_cnes` para não gerar explosão de chaves no Join principal.

### 5.1 Remoção de Duplicidades (Base Estabelecimentos)
Garantimos que cada hospital (CNES) possua apenas uma única linha representativa.

In [15]:
# Remover duplicidades do ST
df_st = df_st.drop_duplicates(subset=['codigo_cnes'], keep='last')

### 5.2 Agregação de Leitos (LT)
Os leitos no Datasus geralmente aparecem particionados por tipo (clínico, pediátrico, cirúrgico). Vamos consolidá-los via soma (`sum`) gerando o montante total por CNES.

In [16]:
# Agregação de Leitos
if not df_lt.empty:
    for col in ['quantidade_leitos_existentes', 'quantidade_leitos_sus', 'quantidade_leitos_contratados']:
        if col in df_lt.columns:
            df_lt[col] = pd.to_numeric(df_lt[col], errors='coerce').fillna(0)
    
    # Agrupar e somar por CNES
    df_lt_agg = df_lt.groupby('codigo_cnes', as_index=False).sum(numeric_only=True)
    df_st = pd.merge(df_st, df_lt_agg, on='codigo_cnes', how='left')
    
    print("Leitos somados com sucesso.")

Leitos somados com sucesso.


### 5.3 Agregação de Equipamentos (EQ)
Procedimento análogo aos leitos: somaremos aparelhos (Ressonância, Tomografia etc) por instituição.

In [17]:
# Agregação de Equipamentos
if not df_eq.empty:
    for col in ['quantidade_existente', 'quantidade_em_uso']:
        if col in df_eq.columns:
            df_eq[col] = pd.to_numeric(df_eq[col], errors='coerce').fillna(0)
            
    df_eq_agg = df_eq.groupby('codigo_cnes', as_index=False).sum(numeric_only=True)
    df_st = pd.merge(df_st, df_eq_agg, on='codigo_cnes', how='left')
    
    print("Equipamentos somados com sucesso.")

Equipamentos somados com sucesso.


### 5.4 Agregação de Habilitações e Serviços (HB e SR)
Punções de classificadores sistêmicos da rede. Apenas vincularemos o primeiro registro listado (tags categóricas consolidadas).

In [18]:
# SR e HB - contagem de marcações ou primeira marcação
if not df_sr.empty:
    df_sr_agg = df_sr.groupby('codigo_cnes', as_index=False).first()
    cols_drop_sr = [c for c in df_sr_agg.columns if c in df_st.columns and c != 'codigo_cnes']
    df_st = pd.merge(df_st, df_sr_agg.drop(columns=cols_drop_sr), on='codigo_cnes', how='left')

if not df_hb.empty:
    df_hb_agg = df_hb.groupby('codigo_cnes', as_index=False).first()
    cols_drop_hb = [c for c in df_hb_agg.columns if c in df_st.columns and c != 'codigo_cnes']
    df_st = pd.merge(df_st, df_hb_agg.drop(columns=cols_drop_hb), on='codigo_cnes', how='left')
    
    print("Serviços Especializados anexados com sucesso.")

Serviços Especializados anexados com sucesso.


### 5.5 Preenchimento de Nulos pós-merge numéricos
Como fizemos vários agrupamentos tipo "Left Join", hospitais que não existiam numa planilha específica (Ex: LT) virão como Nulos na nossa tabela. Precisamos zerálos.

In [19]:
# Base Consolidada CNES
df_cnes_final = df_st.copy()

numeric_cols = df_cnes_final.select_dtypes(include='number').columns
df_cnes_final[numeric_cols] = df_cnes_final[numeric_cols].fillna(0)

print(f"Base Mestre CNES estruturada com {df_cnes_final.shape[0]} hospitais únicos.")
df_cnes_final.head(3)

Base Mestre CNES estruturada com 109849 hospitais únicos.


,codigo_cnes,codigo_municipio,cep_estabelecimento,cpf_cnpj_estabelecimento,tipo_pessoa,nivel_dependencia,cnpj_mantenedora,codigo_retencao_mantenedora,codigo_regiao_saude,codigo_micro_regiao_saude,...,indicador_terceirizado,caracterizacao_servico,indicador_servico_unico,codigo_cnes_terceiro,codigo_habilitacao,competencia_inicial,competencia_final,data_portaria,numero_portaria,competencia_portaria
0,0047406,350010,17800037,35723744000119,3,1,00000000000000,NaN,0209,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0081655,350010,17800057,00381929000108,3,1,00000000000000,NaN,R209,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0109789,350010,17803116,36060657000191,3,1,00000000000000,NaN,0209,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 5.6 Contagem de Nulos no CNES Final
Amostrando a proporção de falta de registros nas recém formatadas colunas.

In [20]:
nulos_cnes = pd.DataFrame({
    "qtd_nulos": df_cnes_final.isnull().sum(),
    "perc_nulos": df_cnes_final.isnull().mean() * 100
}).sort_values(by="perc_nulos", ascending=False)

nulos_cnes.head(20)

,qtd_nulos,perc_nulos
codigo_micro_regiao_saude,109849,100.00
codigo_retencao_mantenedora,109849,100.00
codigo_modulo_assistencial,109849,100.00
codigo_nivel_hierarquia,109849,100.00
codigo_retencao_tributaria,109849,100.00
codigo_natureza_organizacao,109849,100.00
data_publicacao_contrato_estadual,109849,100.00
numero_contrato_estadual,109849,100.00
data_publicacao_contrato_municipal,109849,100.00
numero_contrato_municipal,109849,100.00


In [ ]:
# apagar colunas que tenham mais de 50% de nulos

### 5.7 Salvar Master Base de Hospitais (Interim CNES)

In [21]:
path_cnes_interim = INTERIM / "cnes_hospitais.csv"
df_cnes_final.to_csv(path_cnes_interim, index=False)
print(f"CNES Interim salvo em {path_cnes_interim}")

CNES Interim salvo em /home/carolina/Documents/TCC Documentos/TCC/data/interim/cnes_hospitais.csv


## 6. Fusão Global do Universo DATASUS (SIH IAM + CNES Base Master)
Aplica os atributos macro do hospital às trajetórias dos pacientes SIAM.

In [22]:
# 1. Renomear chave no SIH para viabilizar merge seguro
if 'cnes' in df_iam.columns:
    df_iam = df_iam.rename(columns={'cnes': 'codigo_cnes'})
df_iam["codigo_cnes"] = df_iam["codigo_cnes"].astype(str)
df_cnes_final["codigo_cnes"] = df_cnes_final["codigo_cnes"].astype(str)

# 2. Merge Left SIH x CNES
df_base_modelagem = pd.merge(df_iam, df_cnes_final, on="codigo_cnes", how="left")

# 3. Exportação para a malha final
path_base_final = PROCESSED / "base_modelagem.csv"
df_base_modelagem.to_csv(path_base_final, index=False)
print(f"Cruzamento Global concluído com sucesso e exportado em {path_base_final} com features: {len(df_base_modelagem.columns)}")
df_base_modelagem.sample(3)

Cruzamento Global concluído com sucesso e exportado em /home/carolina/Documents/TCC Documentos/TCC/data/processed/base_modelagem.csv com features: 246


,especialidade_leito,numero_aih,data_nascimento,sexo,uti_mes_total,tipo_uti,procedimento_solicitado,procedimento_realizado,data_internacao,data_saida,...,indicador_terceirizado,caracterizacao_servico,indicador_servico_unico,codigo_cnes_terceiro,codigo_habilitacao,competencia_inicial,competencia_final,data_portaria,numero_portaria,competencia_portaria
3070,1,3524132727918,19501020,1,0,0,406030022,406030022,20241206,20241208,...,NaN,1,1,NaN,2902,202306,999999,04/12/2025,PORTARIA SAES/MS NÂº 701/2023,202512
3550,3,3524131619350,19501124,3,0,0,303060190,303060190,20241011,20241031,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1,3524128892394,19551101,1,0,0,406030049,406030049,20241214,20241218,...,NaN,1,1,NaN,1104,200810,999999,14/10/2008,PT SAS 572,200810
